In [53]:
import torch
from torch.utils.data import DataLoader, random_split, Subset
from torchgeo.datasets import RESISC45
from pathlib import Path
from torchvision.datasets import ImageFolder
from torchvision.transforms import ToTensor

In [52]:
ROOT = Path('datageo')

RESISC45(root=str(ROOT), download=True, checksum=True)
image_dir = ROOT / "filter"
dataset = ImageFolder(image_dir, transform=ToTensor())

print("Dataset location:", image_dir.resolve())
print("Images:", len(dataset))
print("Classes:", len(dataset.classes))
print(dataset.classes)


Dataset location: /Users/dagurgudnason/Desktop/Gervigreind/datageo/filter
Images: 1400
Classes: 2
['baseball_diamond', 'basketball_court']


In [66]:
batch_size = 256

train_dataset, test_dataset, val_dataset = random_split(
    dataset,
    [1000, 200, 200],
generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=1024, shuffle=False)

test_loader = DataLoader(
    dataset())

print(f"Training dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")

print(f"Test set size: {len(test_loader.dataset)}")


TypeError: 'ImageFolder' object is not callable

In [48]:

# For checking only; this is not your training loader.
check_loader = DataLoader(dataset, batch_size=32, num_workers=0)

images, labels = next(iter(check_loader))

assert images.shape == (32, 3, 256, 256)
assert images.dtype == torch.float32
assert torch.isfinite(images).all()
assert 0 <= images.min() and images.max() <= 1
assert labels.shape == (32,)
assert labels.dtype == torch.int64
assert ((labels >= 0) & (labels < 45)).all()

print("PASS: batch loads correctly.")
print("Image batch:", images.shape)
print("Pixel range:", images.min().item(), images.max().item())

PASS: batch loads correctly.
Image batch: torch.Size([32, 3, 256, 256])
Pixel range: 0.0 1.0


In [ ]:
# HW7

import torch.nn as nn

class CNN_Classifier(nn.Module):
    def __init__(self, *args, **kwargs):
        super(CNN_Classifier, self).__init__()
        width = 1
        self.features = nn.Sequential(
            nn.Conv2d(3, 32*width, kernel_size= 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(32*width, 32*width, kernel_size= 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0,2),
            
            nn.Conv2d(32*width, 64*width, kernel_size= 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64*width, 64*width, kernel_size= 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0,3),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*8*8*width, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(in_features=128, out_features=2)
        )
        
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [51]:
if torch.cuda.is_available():
    device = torch.device("cuda") # NVIDIA Cuda framework
elif torch.backends.mps.is_available():
    device = torch.device("mps") # MacBook MPS framework
else:
    device = torch.device("cpu")
print("Device: ", device)

Device:  mps


In [ ]:
def train(model, optimizer, n_epochs, loss_history):
    model.train()
    for epoch in range(n_epochs):
        tstart = time.time()
        train_loss = 0 # Track total training loss for each epoch
        model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device) # Move data
        optimizer.zero_grad() # Reset gradient computation to avoid
    # accumulating gradient over multiple b
    output = model(data) # Performs the forward operation
    loss = F.cross_entropy(output, target) # Average loss over th
    train_loss += loss.item() * target.size(0)
    loss.backward() # Compute the gradient of the loss (via auto
    optimizer.step() # Update model parameters (SGD step)
    train_loss /= len(train_loader.dataset)
    # Compute performance on validation set at the end of each epoch
    val_loss, val_error, _, _ = evaluate(model, F.cross_entropy, val_
    print(f'Epoch: {epoch + 1:d}. Training loss: {train_loss:.4f}, '
    f'validation loss: {val_loss:.4f}, ' +
    f'validation error: {val_error:.4f} ' +
    f'({(time.time()-tstart):.2f} seconds)')
    loss_history['train_loss'].append(train_loss)
    loss_history['val_loss'].append(val_loss)
    loss_history['val_error'].append(val_error)
    return model

In [ ]:
import time
import torch.optim as optim
import torch.nn.functional as F
model = CNN_Classifier()
model.to(device) # move model to GPU (if available)
# Main training run
n_epochs = 20
optimizer = optim.SGD(model.parameters(), lr=0.25)
loss_history = {'train_loss' : [], 'val_loss' : [], 'val_error' : []}
model, loss_history = train(model, optimizer, n_epochs, loss_history)

- overfitta á mjög litlu datasetti
- resnet
- clip


_______

In [1]:
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split

In [ ]:
# chat model...
IMG_SIZE = 224

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
DATA_DIR = "datageo/filter"

# Load dataset
dataset = datasets.ImageFolder(DATA_DIR)

labels = np.array(dataset.targets)
indices = np.arange(len(dataset))

# 80% train, 20% remaining
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

# Split remaining 20% into 4% validation and 16% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.80,
    stratify=labels[temp_idx],
    random_state=42
)

# Create datasets
train_dataset = Subset(
    datasets.ImageFolder(DATA_DIR, transform=train_transform),
    train_idx
)

val_dataset = Subset(
    datasets.ImageFolder(DATA_DIR, transform=test_transform),
    val_idx
)

test_dataset = Subset(
    datasets.ImageFolder(DATA_DIR, transform=test_transform),
    test_idx
)

In [15]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4
)

In [16]:
print(f"Train: {len(train_dataset)} images")
print(f"Validation: {len(val_dataset)} images")
print(f"Test: {len(test_dataset)} images")

images, labels = next(iter(train_loader))

print(f"\nBatch shape: {images.shape}")
print(f"Labels shape: {labels.shape}")

Train: 1120 images
Validation: 56 images
Test: 224 images

Batch shape: torch.Size([32, 3, 256, 256])
Labels shape: torch.Size([32])


In [ ]:
#RESNET18
model = torchvision.models.resnet18()